In [1]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset（180次元：d, o, acc, d1, d2, f3, f5, f11, diff） --------
class RelativeSpeedDataset180D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            def smooth(x, w):
                return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items:
                    return

                d = dist[i:i+20]
                o = own[i:i+20]
                t = tgt[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)) or np.any(np.isnan(t)):
                    continue

                rel_speed = t - o
                acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)
                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f11 = smooth(d, 11)
                diff = np.diff(d, prepend=d[0])

                try:
                    feat = np.concatenate([
                        d, o, acc, d1, d2,
                        f3[:20], f5[:20], f11[:20], diff[:20]
                    ])
                except:
                    continue

                if feat.shape[0] != 180:
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return torch.tensor(feat), torch.tensor(tgt, dtype=torch.float32), sid

# -------- 最終Attention付きLSTMモデル --------
class AttnLSTMv3Model(nn.Module):
    def __init__(self, input_dim=9, hidden_dim=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

        self.attn_fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.view(x.size(0), 20, 9)         # (B, 20, 9)
        lstm_out, _ = self.lstm(x)           # (B, 20, hidden)
        attn_weights = torch.softmax(self.attn_fc(lstm_out), dim=1)  # (B, 20, 1)
        context = (lstm_out * attn_weights).sum(dim=1)               # (B, hidden)
        return self.fc(context).squeeze(1)

# -------- 学習ループ --------
def train_attn_lstm_v3(dataset, save_path="model_attn_lstm_v3.pth"):
    scenes = sorted(set([item[-1] for item in dataset.items]))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)
    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    def collate_fn(batch):
        feats, tgts, _ = zip(*batch)
        return torch.stack(feats), torch.tensor(tgts)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AttnLSTMv3Model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.SmoothL1Loss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_val_loss = float('inf')
    patience = 30
    min_delta = 0.0002
    counter = 0

    for epoch in range(100):
        model.train()
        total_train_loss = 0
        for feats, tgts in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            optimizer.zero_grad()
            loss = criterion(model(feats), tgts)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                loss = criterion(model(feats), tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if best_val_loss - val_loss > min_delta:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved model to {save_path} (val_loss={val_loss:.4f})")
            counter = 0
        else:
            counter += 1
            print(f"⏸ No improvement. Patience: {counter}/{patience}")
            if counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- 実行 --------
if __name__ == "__main__":
    dataset = RelativeSpeedDataset180D(
        annot_root="../train/train_annotations",
        distance_json_path="../train2/distance1/corrected_distance_estimates_filtered.json",
        max_items=7500
    )

    model = train_attn_lstm_v3(dataset, save_path="model_attn_lstm_v3.pth")
    print("✅ 学習完了: model_attn_lstm_v3.pth に保存しました")


[Train 1]: 100%|██████████| 92/92 [00:00<00:00, 94.73it/s] 


Epoch 1 | Train Loss: 2.9902 | Val Loss: 0.5951
✅ Saved model to model_attn_lstm_v3.pth (val_loss=0.5951)


[Train 2]: 100%|██████████| 92/92 [00:00<00:00, 156.64it/s]


Epoch 2 | Train Loss: 0.3850 | Val Loss: 0.2207
✅ Saved model to model_attn_lstm_v3.pth (val_loss=0.2207)


[Train 3]: 100%|██████████| 92/92 [00:00<00:00, 159.51it/s]


...

Epoch 82 | Train Loss: 0.0554 | Val Loss: 0.0443
⏸ No improvement. Patience: 28/30


[Train 83]: 100%|██████████| 92/92 [00:00<00:00, 157.28it/s]


Epoch 83 | Train Loss: 0.0565 | Val Loss: 0.0450
⏸ No improvement. Patience: 29/30


[Train 84]: 100%|██████████| 92/92 [00:00<00:00, 155.58it/s]


Epoch 84 | Train Loss: 0.0539 | Val Loss: 0.0444
⏸ No improvement. Patience: 30/30
🛑 Early stopping at epoch 84
✅ 学習完了: model_attn_lstm_v3.pth に保存しました
